# 710.1x: Multi-layer 1D flow solutions


This notebook demonstrates Bruggeman's multi-layer analytical solutions for one-dimensional flow (710 series).
These solutions model flow in systems with multiple aquifers separated by semi-pervious layers.

Solutions demonstrated:
- Bruggeman 710.12: Steady-state drawdown from surface water level change

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import timflow.steady as tfs

from bruggeman import bruggeman_710_12
from bruggeman.multilayer.funcs import get_psi_for_contour, get_phi_for_contour

## Bruggeman 710.12: Steady-state sudden drawdown

All aquifers with open boundary. Sudden drawdown of the surface water level, which is
kept constant thereafter, $\phi = \phi(x) = \text{drawdown}$. Note that this function
returns both the head $\varphi$ and the flow $Q_x$.

In [ ]:
bruggeman_710_12

In [ ]:
help(bruggeman_710_12)

Compute an example with 5 layers, and compare it with timflow.

In [ ]:
# Parameters
b = 50  # maximum distance [m]
nlay = 5  # number of layers

# Layer properties
d = np.array([5, 10, 5, 10, 2])  # aquitard thickness
D = np.array([15, 20, 15, 20, 20])  # aquifer thickness [m]
k = np.array([2, 2, 5, 10, 15])  # conductivity [m/d]
c = np.array([100, 10, 200, 40, 100, np.inf])  # resistance [d]
h = 1 * np.ones_like(D)  # drawdown [m]

# Build Z array (elevation levels)
zstart = 0.0
def z_from_thicknesses(Hll, Haq, zstart=0.0):
    """Compute elevation levels from aquifer and aquitard thicknesses."""
    z = np.concatenate([[zstart], -np.cumsum(list(zip(Hll, Haq)))])
    return z
z = z_from_thicknesses(d, D, zstart=0.0)

# x array
x = np.linspace(0, b, 101)

# Compute solution
phi, Qx = bruggeman_710_12(x, h, k, D, c)

Compute timflow solution.

In [ ]:
ml = tfs.ModelXsection(naq=nlay)
tfs.XsectionMaq(ml, x1=-np.inf, x2=np.inf, z=z, kaq=k, c=c[:-1], topboundary="semi", hstar=0.0)
tfs.River1D(ml, xls=0.0, hls=1.0, layers=list(range(nlay)))
ml.solve(silent=True)
h_ml = ml.headalongline(x[::10], np.zeros_like(x[::10]))

Plot comparison between Bruggeman solution and timflow.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3))
for i in range(nlay):
    p1, = ax.plot(x, phi[i, :], label=f'Layer {i}')
    ax.plot(x[::10], h_ml[i], marker="x", color=p1.get_color(), ls="none")
ax.set_xlabel('x [m]')
ax.set_ylabel('head [m]')
ax.plot([], [], "kx", label='timflow')
ax.legend(loc=(0, 1), frameon=False, ncol=6, fontsize='small')
ax.set_title("Bruggeman 710.12 - head", y=1.15)
ax.grid()

Compare the stream function between bruggeman and timflow solutions.

In [ ]:
psi = get_psi_for_contour(Qx)

fig, ax = plt.subplots(figsize=(10, 3))
# ax.set_aspect("equal", adjustable="box")
ax.contour(x, z, psi, colors="C0", levels=20)
ax = ml.plots.vcontour_stream_function(
    x1=0, x2=50, nx=20, levels=20, color="C1", ax=ax, linestyles="dashed"
)
ax.set_title("Bruggeman 710.12 - stream function", y=1.10)
ax.plot([], [], "C0", label="Bruggeman 710.12")
ax.plot([], [], "C1", ls="dashed", label="timflow")
ax.legend(loc=(0, 1), frameon=False, ncol=2, fontsize="small");

In [ ]:
# contour heads in vertical cross section
# phi_full = get_phi_for_contour(phi)
# f, ax = plt.subplots(figsize=(5, 5))
# ax.set_aspect("equal", adjustable="box")
# ax.contour(x, z[1:], phi_full, colors="C1", levels=20)
# ml.plots.vcontour(
#     [0, 50, 0, 0],
#     n=20,
#     levels=20,
#     labels=False,
#     vinterp=False,
#     color="darkred",
#     linestyles="dashed",
#     ax=ax,
# )